# Deploying Iris-detection model using Vertex AI


## Overview

In this tutorial, you build a scikit-learn model and deploy it on Vertex AI using the custom container method. You use the FastAPI Python web server framework to create a prediction endpoint. You also incorporate a preprocessor from training pipeline into your online serving application.

Learn more about [Custom training](https://cloud.google.com/vertex-ai/docs/training/custom-training) and [Vertex AI Prediction](https://cloud.google.com/vertex-ai/docs/predictions/get-predictions).

### Objective

In this notebook, you learn how to create, deploy and serve a custom classification model on Vertex AI. This notebook focuses more on deploying the model than on the design of the model itself. 


This tutorial uses the following Vertex AI services and resources:

- Vertex AI models
- Vertex AI endpoints

The steps performed include:

- Train a model that uses flower's measurements as input to predict the class of iris.
- Save the model and its serialized pre-processor.
- Build a FastAPI server to handle predictions and health checks.
- Build a custom container with model artifacts.
- Upload and deploy custom container to Vertex AI Endpoints.

### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Costs 

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage
* Artifact Registry
* Cloud Build

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), [Artifact Registry pricing](https://cloud.google.com/artifact-registry/pricing) and [Cloud Build pricing](https://cloud.google.com/build/pricing) and use the [Pricing
Calculator](https://cloud.google.com/products/calculator/)
to generate a cost estimate based on your projected usage.

## Get started

### Install Vertex AI SDK for Python and other required packages



In [ ]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information 
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
PROJECT_ID = "arcane-rigging-461217-m1"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [ ]:
BUCKET_URI = f"gs://mlops-course-arcane-rigging-461217-m1"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [ ]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com). 

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [ ]:
import os
import sys

### Configure resource names

Set a name for the following parameters:

`MODEL_ARTIFACT_DIR` - Folder directory path to your model artifacts within a Cloud Storage bucket, for example: "my-models/fraud-detection/trial-4"

`REPOSITORY` - Name of the Artifact Repository to create or use.

`IMAGE` - Name of the container image that is pushed to the repository.

`MODEL_DISPLAY_NAME` - Display name of Vertex AI model resource.

In [ ]:
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-week-1"  # @param {type:"string"}
REPOSITORY = "iris-classifier-repo"  # @param {type:"string"}
IMAGE = "iris-classifier-img"  # @param {type:"string"}
MODEL_DISPLAY_NAME = "iris-classifier"  # @param {type:"string"}

# Set the defaults if no names were specified
if MODEL_ARTIFACT_DIR == "[your-artifact-directory]":
    MODEL_ARTIFACT_DIR = "custom-container-prediction-model"

if REPOSITORY == "[your-repository-name]":
    REPOSITORY = "custom-container-prediction"

if IMAGE == "[your-image-name]":
    IMAGE = "sklearn-fastapi-server"

if MODEL_DISPLAY_NAME == "[your-model-display-name]":
    MODEL_DISPLAY_NAME = "sklearn-custom-container"

## Simple Decision Tree model
Build a Decision Tree model on iris data using features served from Feast feature store.

### Model Training

In [1]:
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")
%env PYTHONWARNINGS=ignore
%env JUPYTER_PLATFORM_DIRS=1

env: PYTHONWARNINGS=ignore
env: JUPYTER_PLATFORM_DIRS=1


In [2]:
# Import the necessary libraries

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

from feast import FeatureStore

import joblib

In [3]:
# Initialize the Feast feature store
# This connects to the pre-configured feature repository
store = FeatureStore(repo_path="Iris_Feast/feature_repo")

/home/jupyter/assignment/.env/lib/python3.12/site-packages/feast/repo_config.py:268: DeprecationWarning: The serialization version 2 and below will be deprecated in the next release. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


In [4]:
# Load entity dataframe with timestamps for historical feature retrieval
entity_df = pd.read_csv("data/entity.csv", parse_dates=['event_timestamp'])

# Retrieve historical features using Feast
# This gets features at specific points in time for training
data = store.get_historical_features(
    entity_df=entity_df,
    features=store.get_feature_service("feast_model_v1")
).to_df()

# Display first 10 rows of the dataset
print("First 10 rows of the dataset:")
data.head(10)

First 10 rows of the dataset:


,species,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
0,setosa,2025-06-21 19:27:06.043098+00:00,4.9,3.0,1.4,0.2
1,setosa,2025-06-21 22:37:06.043098+00:00,5.1,3.4,1.5,0.2
2,setosa,2025-06-21 23:17:06.043098+00:00,4.6,3.2,1.4,0.2
3,setosa,2025-06-21 20:12:06.043098+00:00,5.4,3.7,1.5,0.2
4,setosa,2025-06-21 21:17:06.043098+00:00,5.1,3.3,1.7,0.5
5,setosa,2025-06-21 21:27:06.043098+00:00,5.0,3.0,1.6,0.2
6,setosa,2025-06-21 21:52:06.043098+00:00,4.8,3.1,1.6,0.2
7,setosa,2025-06-21 22:57:06.043098+00:00,5.0,3.5,1.6,0.6
8,setosa,2025-06-21 22:07:06.043098+00:00,5.5,4.2,1.4,0.2
9,setosa,2025-06-21 23:22:06.043098+00:00,5.3,3.7,1.5,0.2


In [5]:
# Get information about the dataset structure
print("Dataset Information:")
data.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   species          147 non-null    object             
 1   event_timestamp  147 non-null    datetime64[us, UTC]
 2   sepal_length     147 non-null    float64            
 3   sepal_width      147 non-null    float64            
 4   petal_length     147 non-null    float64            
 5   petal_width      147 non-null    float64            
dtypes: datetime64[us, UTC](1), float64(4), object(1)
memory usage: 7.0+ KB


In [6]:
# Split data into training and testing sets
# Using stratified split to maintain class distribution
train, test = train_test_split(
    data, 
    test_size=0.4,  # 40% for testing
    stratify=data['species'],  # Maintain class proportions
    random_state=42  # For reproducible results
)

# Prepare feature matrices and target vectors
# Features: sepal and petal measurements
X_train = train[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_train = train.species

X_test = test[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_test = test.species

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 88
Test set size: 59


In [7]:
# Initialize Decision Tree classifier
mod_dt = DecisionTreeClassifier(
    max_depth=3, 
    random_state=1  # For reproducible results
)

# Train the model
mod_dt.fit(X_train, y_train)
print("Model training completed!")

Model training completed!


In [8]:
# Make predictions on test set
prediction = mod_dt.predict(X_test)

# Calculate and display accuracy
accuracy = metrics.accuracy_score(y_test, prediction)
print(f'The accuracy of the Decision Tree is {accuracy:.3f}')

The accuracy of the Decision Tree is 0.915


In [9]:
# Save the trained model for future use
joblib.dump(mod_dt, "artifacts/model.joblib")
print("Model saved successfully to artifacts/model.joblib")

Model saved successfully to artifacts/model.joblib


### Online inferencing using the trained model

In [10]:
# Load the trained model
model = joblib.load("artifacts/model.joblib")

In [11]:
# Demonstrate online feature serving
# Get the latest features for each species
features = store.get_online_features(
    features=store.get_feature_service("feast_model_v1"),
    entity_rows=[
        {"species": "setosa"},
        {"species": "versicolor"}, 
        {"species": "virginica"}
    ],
).to_df()

print("Online features retrieved:")
features

Online features retrieved:


,species,petal_length,petal_width,sepal_width,sepal_length
0,setosa,1.4,0.2,3.3,5.0
1,versicolor,4.1,1.3,2.8,5.7
2,virginica,5.1,1.8,3.0,5.9


In [12]:
# Apply trained model to online features
features["model_prediction"] = model.predict(
    features[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
)

print("Features with model predictions:")
features

Features with model predictions:


,species,petal_length,petal_width,sepal_width,sepal_length,model_prediction
0,setosa,1.4,0.2,3.3,5.0,setosa
1,versicolor,4.1,1.3,2.8,5.7,versicolor
2,virginica,5.1,1.8,3.0,5.9,virginica


### Upload model artifacts and custom code to Cloud Storage

Before you can deploy your model for serving, Vertex AI needs access to the following files in Cloud Storage:

* `model.joblib` (model artifact)
* `preprocessor.pkl` (model artifact)

Run the following commands to upload your files:

In [ ]:
!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/